In [0]:
import pandas as pd
from datetime import date
from pyspark.sql.functions import col, lit, when

Global


In [0]:
tb_moni = 'ia.tb_diamond_mod_monitoramento'

tb_entrada = 'ia.tb_diamond_mod_reumatologia'
tb_saida = 'ia.tb_diamond_mod_reumatologia_saida'

Criacao da tabela de monitoramento

In [0]:
df_entrada = spark.sql(f"""
    SELECT
        cast(dataExecucaoModelo as date) as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(an) AS qtd_entrada,
        unidade as nm_unidade,
        idunidade as id_unidade
    FROM {tb_entrada}
    GROUP BY dataExecucaoModelo, nme_regional_hospital, unidade, idunidade
    ORDER BY dataExecucaoModelo
""")
df_saida = spark.sql(f"""
    SELECT
        dataExecucaoModelo as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(exm_an) AS qtd_saida,
        unidade as nm_unidade,
        idunidade as id_unidade
    FROM {tb_saida}
    GROUP BY dataExecucaoModelo, nme_regional_hospital,unidade, idunidade
    ORDER BY dataExecucaoModelo
""")

df_envio = spark.sql(f"""
    SELECT
        dataExecucaoModelo as dt_execucao,
        nme_regional_hospital as nm_regional,
        COUNT(exm_an) AS qtd_envio,
        unidade as nm_unidade,
        idunidade AS id_unidade
    FROM {tb_saida}
    WHERE fl_relevante = 1
    GROUP BY dataExecucaoModelo, nme_regional_hospital,unidade, idunidade
    ORDER BY dataExecucaoModelo
""")

In [0]:
df_monitoramento = (
    df_entrada.alias("e")
        .join(df_saida.alias("s"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .join(df_envio.alias("v"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .select(
            "e.dt_execucao",
            "e.qtd_entrada",
            "s.qtd_saida",
            "v.qtd_envio",
            "e.nm_regional",
            "e.nm_unidade",
        )
)

In [0]:
df_monitoramento = (
    df_monitoramento
        .withColumn("qtd_entrada", col("qtd_entrada").cast("int"))
        .withColumn("qtd_saida", col("qtd_saida").cast("int"))
        .withColumn("qtd_envio", col("qtd_envio").cast("int"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_projeto", lit("Reumatologia").cast("string"))
)

In [0]:
df_monitoramento.createOrReplaceTempView("vw_monitoramento_wrk_01")

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_wrk_01 s
ON  t.dt_execucao = s.dt_execucao
AND t.nm_regional = s.nm_regional
AND t.nm_unidade = s.nm_unidade
AND t.nm_projeto  = s.nm_projeto

WHEN NOT MATCHED THEN
  INSERT (
    dt_execucao,
    qtd_entrada,
    qtd_saida,
    qtd_envio,
    nm_regional,
    nm_unidade,
    nm_projeto
    
  )
  VALUES (
    s.dt_execucao,
    s.qtd_entrada,
    s.qtd_saida,
    s.qtd_envio,
    s.nm_regional,
    s.nm_unidade,
    s.nm_projeto
  )

In [0]:
dbutils.notebook().exit("Fim da execução!")